In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup

In [ ]:
service = Service("/usr/local/bin/chromedriver") # r".....\chromedriver.exe"
chrome_options = Options()
driver = webdriver.Chrome(service=service, options=chrome_options)

In [ ]:
pages = []

for i in range(1, 42):
    pages.append(f"https://www.alza.cz/chytre-hodinky-smartwatch/18854785.htm?setlang=cs-CZ#f&cst=0,2,3,1&cud=0&pg={i}&prod=")

In [ ]:
product_list = []
driver.get("https://www.alza.cz/chytre-hodinky-smartwatch/18854785.htm?setlang=cs-CZ#f&cst=0,2,3,1&cud=0&pg=1&prod=")
links = driver.find_elements(By.XPATH, "//a[@class='name browsinglink js-box-link']")
for link in links:
    product_list.append(link.get_attribute("href"))




for page in pages:
    driver.get(page)
    links = driver.find_elements(By.XPATH, "//a[@class='name browsinglink js-box-link']")
    for link in links:
        product_list.append(link.get_attribute("href"))


In [ ]:
# Dataset o produktech
title = []
description = []
price = []
rating = []
review_count = []
image_link = []
page = []

# Dataset pro modelovani sentimentu
reviews = []
ratings = []
link = []

for product in product_list:

    driver.get(product)

    title.append(driver.find_element(By.XPATH, "//h1[@class='h1-placeholder']").text)
    description.append(driver.find_element(By.XPATH, "//div[@class='nameextc']/span").text)
    price.append(driver.find_element(By.XPATH, "//span[@class='price-box__price-text']/span[@class='price-box__price']").text)
    rating.append(driver.find_element(By.XPATH, "//span[@class='ratingValue']").text)
    review_count.append(driver.find_element(By.XPATH, "//span[@class='ratingCount']").text)
    image_link.append(driver.find_element(By.XPATH, "//img[@class='detailGallery-alz-31 detailGallery-alz-25']").get_attribute("src"))
    page.append(product)

    reviews_button = driver.find_element(By.ID, "reviews-tab")
    driver.execute_script("arguments[0].click();", reviews_button)
    time.sleep(1)

    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")

    review_list_2 = soup.find_all("div", class_="reviewsTab-alz-7")
    for review in review_list_2:
        try:
            reviews.append(review.find("div", class_="reviewsTab-alz-120").find("span", class_="AlzaText reviewsTab-alz-67 reviewsTab-alz-76").text) # element s uzivatelem napsanou recenzi
            ratings.append(re.search(r"[0-9]{2,3}", review.find("span", class_="reviewsTab-alz-44").get("style")).group()) # element s ratingem –> *****
            link.append(product)
        except Exception:
            continue


In [ ]:
products_dataset = {
    "page": page,
    "title": title,
    "description": description,
    "price": price,
    "rating": rating,
    "review_count": review_count,
    "image": image_link
}

reviews_dataset = {
    "review": reviews,
    "rating": ratings,
    "url": link
}

products_dataset = pd.DataFrame(products_dataset)
reviews_dataset = pd.DataFrame(reviews_dataset)

In [ ]:
products_dataset.to_csv("")
reviews_dataset.to_csv("")